In [2]:
import torch

from datasets import load_dataset
from torch.utils.data import Dataset

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

In [5]:
dataset = load_dataset(
    "imagefolder",
    data_dir=r"C:\Users\praut\project crop doctor\sugarcane"
)


Resolving data files:   0%|          | 0/300 [00:00<?, ?it/s]

In [6]:
dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

print("Training:", len(dataset["train"]))
print("Validation:", len(dataset["test"]))

Training: 240
Validation: 60


In [7]:
labels = dataset["train"].features["label"].names

print("Sugarcane classes:")

for i, label in enumerate(labels):
    print(i, ":", label)

id2label = {
    i: label for i, label in enumerate(labels)
}

label2id = {
    label: i for i, label in enumerate(labels)
}

Sugarcane classes:
0 : Bacterial Blight
1 : Healthy
2 : Red Rot


In [8]:
model_name = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(
    model_name
)

print("Processor loaded successfully!")

Processor loaded successfully!


In [9]:
class SugarcaneDataset(Dataset):

    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        image = item["image"].convert("RGB")

        inputs = self.processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": item["label"]
        }

In [10]:
train_dataset = SugarcaneDataset(
    dataset["train"],
    processor
)

eval_dataset = SugarcaneDataset(
    dataset["test"],
    processor
)

print("Sugarcane datasets created!")

Sugarcane datasets created!


In [11]:
sample = train_dataset[0]

print(sample.keys())
print("Image shape:", sample["pixel_values"].shape)
print("Label:", sample["labels"])

dict_keys(['pixel_values', 'labels'])
Image shape: torch.Size([3, 224, 224])
Label: 0


In [12]:
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("Sugarcane ViT model created!")

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.bias             | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.weight | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.bia

Sugarcane ViT model created!


In [13]:
def collate_fn(examples):

    pixel_values = torch.stack([
        example["pixel_values"]
        for example in examples
    ])

    labels_batch = torch.tensor([
        example["labels"]
        for example in examples
    ])

    return {
        "pixel_values": pixel_values,
        "labels": labels_batch
    }

In [14]:
training_args = TrainingArguments(
    output_dir="./sugarcane-vit-results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    logging_steps=50,

    report_to="none"
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    data_collator=collate_fn
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.764907
2,0.773188,0.870166
3,0.773188,0.771693


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=90, training_loss=0.6792744954427083, metrics={'train_runtime': 481.6338, 'train_samples_per_second': 1.495, 'train_steps_per_second': 0.187, 'total_flos': 5.579473258856448e+16, 'train_loss': 0.6792744954427083, 'epoch': 3.0})

In [17]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch
0.773188,0.764907,3


{'eval_loss': 0.7649068236351013}


In [18]:
trainer.save_model("./sugarcane-vit-final")
processor.save_pretrained("./sugarcane-vit-final")

print("🌱 Sugarcane model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🌱 Sugarcane model saved successfully!
